# 3. 토픽 모델링 실습하기

이 노트북은 뉴스 데이터를 불러온 뒤, 한국어 형태소 분석으로 주요 단어를 추출하고, `CountVectorizer`와 `LDA(Latent Dirichlet Allocation)`를 사용하여 문서 안에 숨어 있는 주제를 찾는 실습입니다.


## 1. 실습 데이터 업로드 준비

 Google Colab 환경에서 `news.csv` 파일을 업로드하기 위한 준비 단계입니다. `data` 폴더를 만들고, 작업 위치를 해당 폴더로 이동한 뒤 파일 업로드 창을 실행합니다. 업로드가 끝나면 다시 상위 폴더로 돌아옵니다.

In [1]:
# Google Colab에서 로컬 PC의 파일을 업로드할 수 있게 해 주는 files 모듈을 불러옵니다.
from google.colab import files

# 운영체제의 폴더 생성, 경로 이동, 파일 존재 여부 확인 등에 사용하는 os 모듈을 불러옵니다.
import os

# 업로드한 데이터 파일을 저장할 폴더 이름을 문자열로 지정합니다.
data_dir = 'data'

# 현재 작업 위치에 data 폴더가 존재하지 않는지 확인합니다.
if not os.path.exists(data_dir):
    # data 폴더가 없다면 새로 생성합니다.
    os.mkdir(data_dir)

# 파일을 data 폴더 안에 저장하기 위해 현재 작업 위치를 data 폴더로 변경합니다.
os.chdir(data_dir)

# Colab 파일 업로드 창을 실행하여 사용자가 news.csv 파일을 업로드할 수 있게 합니다.
files.upload()

# 이후 코드에서 ./data/news.csv 경로로 파일을 읽을 수 있도록 작업 위치를 다시 상위 폴더로 변경합니다.
os.chdir('..')

Saving news.csv to news.csv


## 2. 뉴스 CSV 파일 불러오기

업로드한 `news.csv` 파일을 pandas DataFrame으로 읽어옵니다. DataFrame은 표 형태 데이터를 다루기 위한 pandas의 핵심 자료구조입니다.

In [2]:
# CSV 파일을 표 형태의 DataFrame으로 다루기 위해 pandas 라이브러리를 불러옵니다.
import pandas as pd

# ./data/news.csv 파일을 읽어서 df_news라는 DataFrame 변수에 저장합니다.
# 이 파일에는 뉴스 본문, 카테고리, 번호 등의 컬럼이 들어 있다고 가정합니다.
df_news = pd.read_csv('./data/news.csv')

# 데이터가 정상적으로 불러와졌는지 확인하기 위해 전체 행과 열의 개수를 출력합니다.
print('데이터 크기:', df_news.shape)

# 데이터의 컬럼 이름을 확인하여 이후 코드에서 사용할 컬럼명을 점검합니다.
print('컬럼 목록:', df_news.columns.tolist())

데이터 크기: (160, 4)
컬럼 목록: ['num', 'category', 'category_name', 'news']


## 3. 특정 카테고리 데이터 미리보기
 `category` 값이 7인 뉴스 데이터 일부를 확인합니다. 토픽 모델링 전에 데이터가 어떤 구조와 내용을 가지는지 확인하는 과정입니다.

In [3]:
# category 컬럼 값이 7인 행만 필터링합니다.
# == 연산자는 category 값이 7인지 비교하여 True/False 결과를 만듭니다.
category_7_news = df_news[df_news.category == 7]

# 필터링된 데이터 중 앞쪽 5개 행을 출력하여 데이터 내용을 빠르게 확인합니다.
category_7_news.head()

,num,category,category_name,news
140,7001,7,IT,인공지능 기반 자연어 처리 기술이 뉴스 서비스와 검색 플랫폼에 빠르게 적용되고 있다...
141,1142,7,IT,클라우드 플랫폼 기업들은 보안 기능과 소프트웨어 개발 도구를 강화하고 있다. 이번 ...
142,1143,7,IT,반도체 업계는 인공지능 연산에 최적화된 칩과 데이터 처리 기술을 개발하고 있다. 이...
143,1144,7,IT,로봇 서비스는 알고리즘 고도화와 센서 데이터 학습을 통해 활용 범위를 넓히고 있다....
144,1145,7,IT,인공지능 서비스가 다양한 산업에 적용되면서 데이터 분석과 모델 성능 개선이 중요해졌...


## 4. 한국어 형태소 분석 라이브러리 설치

 한국어 자연어 처리를 위해 `JPype1`과 `konlpy`를 설치합니다. KoNLPy의 Okt 형태소 분석기를 사용하려면 Java 연동 라이브러리인 JPype1이 필요합니다.

In [4]:
# KoNLPy가 Java 기반 형태소 분석기를 사용할 수 있도록 JPype1 패키지를 설치합니다.
!pip install JPype1

# 한국어 형태소 분석을 수행하기 위한 KoNLPy 패키지를 설치합니다.
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 51.9 MB/s eta 0:00:00


## 5. 분석할 뉴스 본문 하나 선택하기

특정 카테고리와 번호 조건에 맞는 뉴스 본문 하나를 선택하여 출력합니다. 형태소 분석 결과를 확인하기 위해 먼저 샘플 문장을 준비하는 단계입니다.

In [5]:
# category가 7이고 num이 7001인 뉴스의 news 컬럼 값을 선택합니다.
# values[0]은 조건에 맞는 결과 중 첫 번째 뉴스 본문 문자열을 가져옵니다.
text = df_news[(df_news.category == 7) & (df_news.num == 7001)]['news'].values[0]

# 선택한 뉴스 본문이 어떤 내용인지 확인하기 위해 출력합니다.
print(text)

인공지능 기반 자연어 처리 기술이 뉴스 서비스와 검색 플랫폼에 빠르게 적용되고 있다. 기업들은 대규모 데이터를 학습한 모델을 활용해 문서 분류, 요약, 추천 서비스를 개선하고 있다. 특히 토픽 모델링과 단어 임베딩 기술은 기사 속 핵심 주제를 찾고 관련 뉴스를 묶는 데 사용된다. 보안과 개인정보 보호 문제도 함께 논의되며 클라우드 환경에서 안정적인 데이터 관리가 요구된다.


## 6. 형태소 분석과 정규표현식 모듈 불러오기

한국어 형태소 분석에 사용할 `konlpy`와 텍스트 정제에 사용할 수 있는 `re` 모듈을 불러옵니다.

In [6]:
# 한국어 형태소 분석기를 사용하기 위해 konlpy 라이브러리를 불러옵니다.
import konlpy

# 정규표현식을 이용해 특수문자 제거, 패턴 검색 등을 할 수 있도록 re 모듈을 불러옵니다.
import re

## 7. Okt 형태소 분석기로 품사 확인하기

Okt 형태소 분석기를 생성하고, 선택한 뉴스 본문을 단어와 품사 단위로 분리합니다. `stem=True`는 동사와 형용사를 기본형으로 바꾸는 옵션입니다.

In [7]:
# KoNLPy에서 제공하는 Okt 형태소 분석기 객체를 생성합니다.
okt = konlpy.tag.Okt()

# 선택한 뉴스 본문 text를 형태소 단위로 분리하고 각 형태소의 품사를 함께 반환합니다.
# stem=True는 '했다', '합니다' 같은 표현을 '하다'처럼 기본형으로 정규화합니다.
morphs = okt.pos(text, stem=True)

# 형태소 분석 결과를 출력하여 단어와 품사가 어떻게 분리되었는지 확인합니다.
print(morphs)

[('인공', 'Noun'), ('지능', 'Noun'), ('기반', 'Noun'), ('자연어', 'Noun'), ('처리', 'Noun'), ('기술', 'Noun'), ('이', 'Josa'), ('뉴스', 'Noun'), ('서비스', 'Noun'), ('와', 'Josa'), ('검색', 'Noun'), ('플랫폼', 'Noun'), ('에', 'Josa'), ('빠르다', 'Adjective'), ('적용', 'Noun'), ('되다', 'Verb'), ('있다', 'Adjective'), ('.', 'Punctuation'), ('기업', 'Noun'), ('들', 'Suffix'), ('은', 'Josa'), ('대규모', 'Noun'), ('데이터', 'Noun'), ('를', 'Josa'), ('학습', 'Noun'), ('한', 'Josa'), ('모델', 'Noun'), ('을', 'Josa'), ('활용', 'Noun'), ('하다', 'Verb'), ('문서', 'Noun'), ('분류', 'Noun'), (',', 'Punctuation'), ('요약', 'Noun'), (',', 'Punctuation'), ('추천', 'Noun'), ('서비스', 'Noun'), ('를', 'Josa'), ('개선', 'Noun'), ('하고', 'Josa'), ('있다', 'Adjective'), ('.', 'Punctuation'), ('특히', 'Adverb'), ('토픽', 'Noun'), ('모델링', 'Noun'), ('과', 'Josa'), ('단어', 'Noun'), ('임베딩', 'Noun'), ('기술', 'Noun'), ('은', 'Josa'), ('기사', 'Noun'), ('속', 'Noun'), ('핵심', 'Noun'), ('주제', 'Noun'), ('를', 'Josa'), ('찾다', 'Verb'), ('관련', 'Noun'), ('뉴스', 'Noun'), ('를', 'Josa'), ('묶다', 'Verb'), (

## 8. 뉴스 본문에서 주요 단어만 추출하는 함수 만들기

하나의 뉴스 본문을 입력받아 명사, 형용사, 동사만 추출하는 함수를 정의합니다. 토픽 모델링에서는 조사, 어미, 기호보다 의미 있는 단어가 중요하므로 주요 품사만 남깁니다.

In [8]:
# 하나의 텍스트를 입력받아 토픽 모델링에 사용할 단어 문자열로 변환하는 함수를 정의합니다.
def get_words(text):
    # 입력된 텍스트를 Okt 형태소 분석기로 분석하여 (단어, 품사) 형태의 목록을 만듭니다.
    morphs = okt.pos(text, stem=True)

    # 토픽 모델링에 사용할 단어를 저장할 빈 리스트를 생성합니다.
    word_list = []

    # 형태소 분석 결과에서 단어와 품사를 하나씩 꺼내 반복합니다.
    for word, pos in morphs:
        # 명사, 형용사, 동사는 문서 주제를 파악하는 데 중요한 품사이므로 선택합니다.
        if pos == 'Noun' or pos == 'Adjective' or pos == 'Verb':
            # 한 글자 단어는 의미가 약하거나 노이즈일 가능성이 있으므로 두 글자 이상만 사용합니다.
            if len(word) > 1:
                # 조건을 만족한 단어를 word_list에 추가합니다.
                word_list.append(word)

    # 추출된 단어 리스트를 공백으로 연결하여 CountVectorizer가 처리할 수 있는 문자열로 만듭니다.
    words = ' '.join(word_list)

    # 최종적으로 정제된 단어 문자열을 반환합니다.
    return words

## 9. 전체 뉴스 본문에 단어 추출 함수 적용하기

 `news` 컬럼의 모든 뉴스 본문에 `get_words()` 함수를 적용하여 토픽 모델링용 단어 컬럼인 `words`를 새로 만듭니다.

In [9]:
# df_news의 news 컬럼에 들어 있는 각 뉴스 본문마다 get_words 함수를 적용합니다.
# apply는 각 행의 텍스트를 하나씩 함수에 넣고 결과를 반환합니다.
df_news['words'] = df_news.news.apply(get_words)

# words 컬럼이 제대로 생성되었는지 앞쪽 5개 행으로 확인합니다.
df_news[['news', 'words']].head()

,news,words
0,국회는 새로운 법안 심사를 진행하며 정부 정책 방향을 논의했다. 이번 보도에서는 정...,국회 새롭다 법안 심사 진행 하다 정부 정책 방향 논의 하다 이번 보도 정부 국회 ...
1,정부는 예산 편성과 개혁 과제를 중심으로 부처 회의를 열었다. 이번 보도에서는 정책...,정부 예산 편성 개혁 과제 중심 부처 회의 열다 이번 보도 정책 대통령 정부 법안 ...
2,"대통령은 외교 일정에서 경제 협력과 안보 현안을 강조했다. 이번 보도에서는 개혁, ...",대통령 외교 일정 경제 협력 안보 현안 강조 하다 이번 보도 개혁 정부 회의 국회 ...
3,"선거를 앞두고 각 정당은 공약 발표와 지역 방문을 확대했다. 이번 보도에서는 국회,...",선거 앞두다 정당 공약 발표 지역 방문 확대 하다 이번 보도 국회 정부 정책 개혁 ...
4,국회는 새로운 법안 심사를 진행하며 정부 정책 방향을 논의했다. 이번 보도에서는 개...,국회 새롭다 법안 심사 진행 하다 정부 정책 방향 논의 하다 이번 보도 개혁 국회 ...


## 10. 특정 카테고리의 전처리 결과 확인하기

 category가 7인 데이터에서 원문과 전처리된 단어 결과를 함께 확인합니다. 형태소 분석 결과가 적절한지 점검하는 단계입니다.

In [10]:
# category 값이 7인 뉴스 데이터만 선택합니다.
category_7_news = df_news[df_news.category == 7]

# 선택된 데이터의 앞쪽 5개 행을 출력하여 words 컬럼의 전처리 결과를 확인합니다.
category_7_news.head()

,num,category,category_name,news,words
140,7001,7,IT,인공지능 기반 자연어 처리 기술이 뉴스 서비스와 검색 플랫폼에 빠르게 적용되고 있다...,인공 지능 기반 자연어 처리 기술 뉴스 서비스 검색 플랫폼 빠르다 적용 되다 있다 ...
141,1142,7,IT,클라우드 플랫폼 기업들은 보안 기능과 소프트웨어 개발 도구를 강화하고 있다. 이번 ...,클라우드 플랫폼 기업 보안 기능 소프트웨어 개발 도구 강화하다 있다 이번 보도 디지...
142,1143,7,IT,반도체 업계는 인공지능 연산에 최적화된 칩과 데이터 처리 기술을 개발하고 있다. 이...,반도체 업계 인공 지능 연산 최적화 되다 데이터 처리 기술 개발 있다 이번 보도 보...
143,1144,7,IT,로봇 서비스는 알고리즘 고도화와 센서 데이터 학습을 통해 활용 범위를 넓히고 있다....,로봇 서비스 알고리즘 고도화 센서 데이터 학습 통해 활용 범위 넓히다 있다 이번 보...
144,1145,7,IT,인공지능 서비스가 다양한 산업에 적용되면서 데이터 분석과 모델 성능 개선이 중요해졌...,인공 지능 서비스 다양하다 산업 적용 되다 데이터 분석 모델 성능 개선 중요하다 이...


## 11. CountVectorizer로 문서-단어 행렬 생성 준비하기

 텍스트를 숫자 행렬로 바꾸기 위한 `CountVectorizer` 객체를 생성합니다. LDA는 문자 데이터를 직접 사용할 수 없기 때문에 단어 빈도 기반의 숫자 데이터가 필요합니다.

In [11]:
# 텍스트 데이터를 단어 빈도 행렬로 변환하기 위해 CountVectorizer를 불러옵니다.
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer 객체를 생성합니다.
# max_df=0.1은 전체 문서의 10%를 초과하여 너무 자주 등장하는 단어를 제외한다는 의미입니다.
# max_features=1000은 빈도 기준으로 최대 1000개의 단어 특성만 사용한다는 의미입니다.
# min_df=2는 최소 2개 문서 이상에 등장한 단어만 사용한다는 의미입니다.
# ngram_range=(1, 2)는 한 단어 표현과 두 단어 묶음 표현을 모두 사용한다는 의미입니다.
count_vectorizer = CountVectorizer(
    max_df=0.1,
    max_features=1000,
    min_df=2,
    ngram_range=(1, 2)
)

## 12. 뉴스 단어 데이터를 문서-단어 행렬로 변환하기

전처리된 `words` 컬럼을 CountVectorizer로 학습하고 변환합니다. 결과인 `feat_vect`는 각 문서에 어떤 단어가 몇 번 등장했는지를 담은 희소 행렬입니다.

In [12]:
# words 컬럼의 텍스트 데이터를 학습하여 단어 사전을 만들고, 동시에 문서-단어 빈도 행렬로 변환합니다.
feat_vect = count_vectorizer.fit_transform(df_news.words)

# 변환된 행렬의 크기를 출력합니다.
# 첫 번째 값은 문서 수, 두 번째 값은 선택된 단어 특성 수입니다.
print('문서-단어 행렬 크기:', feat_vect.shape)

# 첫 번째 문서의 단어 빈도 벡터를 출력합니다.
# 대부분의 값이 0인 희소 행렬 형태로 저장됩니다.
print(feat_vect[0, :])

문서-단어 행렬 크기: (160, 697)
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 22 stored elements and shape (1, 697)>
  Coords	Values
  (0, 92)	2
  (0, 319)	1
  (0, 229)	1
  (0, 396)	1
  (0, 570)	1
  (0, 225)	1
  (0, 131)	1
  (0, 333)	1
  (0, 451)	1
  (0, 94)	1
  (0, 320)	1
  (0, 231)	1
  (0, 397)	1
  (0, 571)	1
  (0, 646)	1
  (0, 517)	1
  (0, 522)	1
  (0, 226)	1
  (0, 133)	1
  (0, 282)	1
  (0, 515)	1
  (0, 452)	1


## 13. CountVectorizer가 만든 단어 목록 확인하기

 CountVectorizer가 선택한 단어 특성 이름을 확인합니다. 최신 scikit-learn에서는 `get_feature_names_out()`을 사용합니다.

In [13]:
# CountVectorizer가 학습한 단어 목록을 가져옵니다.
# get_feature_names_out은 최신 scikit-learn에서 사용하는 권장 메서드입니다.
feature_names = count_vectorizer.get_feature_names_out()

# 단어 목록 중 앞쪽 5개를 출력하여 어떤 단어들이 특성으로 사용되는지 확인합니다.
print(feature_names[:5])

['가족' '가족 관련' '가족 중심' '감독' '감독 경기']


## 14. LDA 토픽 모델 생성 및 학습하기

 LDA 모델을 생성하고 문서-단어 행렬을 학습합니다. LDA는 여러 문서 안에 숨어 있는 주제와 각 주제에 자주 등장하는 단어를 찾는 대표적인 토픽 모델링 알고리즘입니다.

In [14]:
# 토픽 모델링 알고리즘인 LatentDirichletAllocation을 불러옵니다.
from sklearn.decomposition import LatentDirichletAllocation

# 추출할 토픽의 개수를 지정합니다.
# 예를 들어 8이면 전체 뉴스 문서를 8개의 숨은 주제로 나누어 보겠다는 의미입니다.
topic_cnt = 8

# LDA 모델 객체를 생성합니다.
# n_components는 찾고자 하는 토픽 개수를 의미합니다.
# random_state는 실행할 때마다 비슷한 결과가 나오도록 난수 시드를 고정합니다.
lda = LatentDirichletAllocation(
    n_components=topic_cnt,
    random_state=42,
    learning_method='batch'
)

# 문서-단어 빈도 행렬을 사용하여 LDA 모델을 학습합니다.
lda.fit(feat_vect)

LatentDirichletAllocation(n_components=8, random_state=42)

## 15. 특정 토픽의 핵심 단어 출력하기

학습된 LDA 모델에서 특정 토픽을 선택한 뒤, 그 토픽에서 가중치가 높은 상위 단어 10개를 출력합니다. 이를 통해 각 토픽이 어떤 의미를 가지는지 사람이 해석할 수 있습니다.

In [15]:
# 확인하고 싶은 토픽 번호를 지정합니다.
# 토픽 번호는 0부터 시작하므로 topic_num=5는 여섯 번째 토픽을 의미합니다.
topic_num = 5

# lda.components_에는 각 토픽별 단어 가중치가 저장되어 있습니다.
# topic 변수에는 선택한 토픽의 모든 단어 가중치가 들어갑니다.
topic = lda.components_[topic_num]

# argsort는 값을 작은 순서대로 정렬했을 때의 인덱스를 반환합니다.
# [::-1]을 붙이면 큰 값부터 작은 값 순서로 인덱스를 뒤집습니다.
topic_word_indexes = topic.argsort()[::-1]

# 가중치가 가장 높은 상위 10개 단어의 인덱스만 선택합니다.
top_indexes = topic_word_indexes[:10]

# 상위 10개 인덱스에 해당하는 단어명을 feature_names에서 찾아 공백으로 연결합니다.
feature_name_list = ' '.join([feature_names[i] for i in top_indexes])

# 선택한 토픽을 대표하는 핵심 단어 목록을 출력합니다.
print(feature_name_list)

경기 훈련 축구 우승 야구 관심 진행 진행 하다 훈련 진행 대표팀 강도


## 16. pyLDAvis 설치하기

 LDA 토픽 모델링 결과를 시각화하기 위한 `pyLDAvis` 패키지를 설치합니다. 이 도구를 사용하면 토픽 간 거리와 토픽별 주요 단어를 시각적으로 확인할 수 있습니다.

In [16]:
# LDA 토픽 모델링 결과를 인터랙티브하게 시각화하기 위한 pyLDAvis 패키지를 설치합니다.
!pip install pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.8 MB/s eta 0:00:00


## 17. pyLDAvis 불러오기 및 노트북 출력 설정하기

pyLDAvis를 불러오고, Jupyter Notebook 또는 Colab 화면 안에서 시각화 결과가 바로 표시되도록 설정합니다.

In [17]:
# pyLDAvis 기본 모듈을 불러옵니다.
import pyLDAvis

# scikit-learn 기반 LDA 모델 결과를 pyLDAvis 형식으로 변환하는 모듈을 불러옵니다.
# pyLDAvis 버전에 따라 sklearn 또는 lda_model 경로가 다를 수 있으므로 예외 처리를 사용합니다.
try:
    # 일부 구버전 pyLDAvis에서 사용하는 import 방식입니다.
    import pyLDAvis.sklearn as sklearn_lda
except ModuleNotFoundError:
    # 최신 pyLDAvis에서 사용하는 import 방식입니다.
    import pyLDAvis.lda_model as sklearn_lda

# 노트북 내부에서 pyLDAvis 시각화 결과가 표시되도록 설정합니다.
pyLDAvis.enable_notebook()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 18. pandas 버전 관련 안내

기존 코드에는 `pandas==1.2`로 강제 다운그레이드하는 코드가 들어 있었지만, 최신 Colab 환경에서는 다른 패키지와 충돌할 수 있습니다. 특별한 오류가 없다면 pandas를 강제로 낮추지 않는 것이 안전합니다. 아래 코드는 현재 pandas 버전만 확인합니다.

In [18]:
# 현재 설치된 pandas 버전을 출력합니다.
# pyLDAvis가 정상 작동한다면 pandas를 강제로 다운그레이드할 필요가 없습니다.
import pandas as pd

# 사용 중인 pandas 버전을 확인합니다.
print('현재 pandas 버전:', pd.__version__)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

현재 pandas 버전: 2.2.2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## 19. LDA 토픽 모델링 결과 시각화하기

학습된 LDA 모델, 문서-단어 행렬, CountVectorizer 정보를 사용하여 pyLDAvis 시각화 객체를 생성합니다. 화면 왼쪽에는 토픽 간 거리가, 오른쪽에는 선택한 토픽의 주요 단어가 표시됩니다.

In [19]:
# 학습된 LDA 모델과 문서-단어 행렬, CountVectorizer를 이용하여 시각화 데이터를 준비합니다.
# sklearn_lda는 pyLDAvis.sklearn 또는 pyLDAvis.lda_model 중 현재 환경에서 사용 가능한 모듈입니다.
vis = sklearn_lda.prepare(lda, feat_vect, count_vectorizer)

# 준비된 pyLDAvis 시각화 결과를 노트북 화면에 표시합니다.
pyLDAvis.display(vis)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 20. 전체 토픽별 핵심 단어 한 번에 확인하기

 8개 토픽 각각에 대해 가중치가 높은 상위 단어를 출력합니다. 특정 토픽 하나만 보는 것보다 전체 토픽의 특징을 비교할 수 있어 결과 해석에 도움이 됩니다.

In [20]:
# 각 토픽에서 출력할 상위 단어 개수를 지정합니다.
top_n = 10

# lda.components_에는 토픽별 단어 가중치가 들어 있으므로 enumerate로 토픽 번호와 가중치를 함께 반복합니다.
for topic_idx, topic_weights in enumerate(lda.components_):
    # 현재 토픽에서 가중치가 큰 단어의 인덱스를 내림차순으로 정렬한 뒤 상위 top_n개만 선택합니다.
    top_word_indexes = topic_weights.argsort()[::-1][:top_n]

    # 선택된 인덱스를 실제 단어 이름으로 변환합니다.
    top_words = [feature_names[i] for i in top_word_indexes]

    # 토픽 번호와 대표 단어들을 출력합니다.
    print(f'Topic {topic_idx}:', ', '.join(top_words))

Topic 0: 인공, 지능, 인공 지능, 반도체, 기술 개발, 에너지, 실험, 되다 데이터, 선거, 처리 기술
Topic 1: 안전, 사회, 플랫폼, 클라우드, 교통, 보안, 우주, 기업 환율, 환율, 소프트웨어
Topic 2: 생활, 소비자, 취미, 주거, 날씨, 중심, 가족, 음식, 여행, 개혁
Topic 3: 사회, 노동, 창작, 문화, 논의, 환경 개선, 커지다, 통해, 요구, 노동 환경
Topic 4: 국회, 법안, 외교, 투자, 소비, 문화, 대통령, 선거, 회복, 모델
Topic 5: 경기, 훈련, 축구, 우승, 야구, 관심, 진행, 진행 하다, 훈련 진행, 대표팀 강도
Topic 6: 선수, 중요하다, 감독, 야구, 리그, 기후, 과학, 경기, 득점, 알고리즘
Topic 7: 전시, 공연, 축제, 영화, 금리, 음악, 관객, 안전, 보건, 복지


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag